# [8.3] ACDC and Circuit Metrics - Solutions

By the end of this notebook, you will have recovered a known two-edge toy circuit with exact edge patching, then checked a small real-model circuit fragment with faithfulness, minimality, completeness, held-out templates, and same-size random controls.

**Core question:** when an automated circuit search keeps an edge, what evidence says this is a real circuit rather than a thresholded metric artifact?

```yaml
gt_tier: GT-1 circuit-metric preflight
exercise_id: 8_3_acdc_and_circuit_metrics
expected_runtime: 60-90 minutes for local exercises; several minutes for the CUDA GELU-1L fragment
requires_gpu: true for the TransformerLens fragment
```

## Learning Objectives

- Implement exact patch-recovery scores from clean, corrupt, and patched metrics.
- Patch named edges in a toy computational graph with known ground truth.
- Prune edges with an ACDC-style threshold and check exact recovery of the known circuit.
- Measure faithfulness, minimality, completeness, OOD robustness, and same-size random controls.
- Compare exact patch scores against approximate circuit-discovery scores.
- Generate a visible circuit table/plot before interpreting a real TransformerLens fragment.

<details>
<summary>Expected output</summary>

The toy graph should recover exactly two edges: `color_input -> answer_logit` and `shape_input -> answer_logit`. The real GELU-1L fragment should show that final-position residual patching recovers the target token, beats wrong-position same-size controls, and passes held-out template checks.

</details>

<details>
<summary>Help - what counts as ACDC-style evidence?</summary>

A thresholded circuit is not enough. You need a known toy oracle first, then a metric battery: the circuit preserves the clean behavior, removing kept edges hurts, adding omitted edges helps little, and same-size random circuits fail.

</details>


In [ ]:
import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import torch as t
from IPython.display import display

chapter = "chapter8_automated_circuits"
section = "part3_acdc_circuit_metrics"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_acdc_circuit_metrics.tests as tests

MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class ActivationPatchingSweep:
    patch_scores: t.Tensor
    best_index: int
    best_score: float


@dataclass(frozen=True)
class ACDCPruningReport:
    kept_edges: tuple[str, ...]
    removed_edges: tuple[str, ...]
    threshold: float
    num_kept: int


@dataclass(frozen=True)
class CircuitFaithfulnessReport:
    full_metric: float
    corrupt_metric: float
    circuit_metric: float
    preserved_fraction: float
    passes_faithfulness: bool


@dataclass(frozen=True)
class CircuitMinimalityReport:
    circuit_metric: float
    ablated_metric: float
    metric_damage: float
    passes_minimality: bool


@dataclass(frozen=True)
class CircuitCompletenessReport:
    circuit_metric: float
    expanded_metric: float
    omitted_node_gain: float
    passes_completeness: bool


@dataclass(frozen=True)
class RandomCircuitBaselineReport:
    circuit_metric: float
    random_metric: float
    margin: float
    circuit_beats_random: bool


@dataclass(frozen=True)
class OODTemplateReport:
    per_template_accuracy: dict[int, float]
    worst_template_accuracy: float
    passes_ood: bool


@dataclass(frozen=True)
class CircuitMethodComparisonReport:
    exact_top_edges: tuple[str, ...]
    method_top_edges: dict[str, tuple[str, ...]]
    topk_overlap: dict[str, float]
    score_correlations: dict[str, float]
    circuit_sizes: dict[str, int]
    best_matching_method: str
    passes_comparison: bool


@dataclass(frozen=True)
class ToyCircuitEvaluationReport:
    ground_truth_edges: tuple[str, ...]
    discovered_edges: tuple[str, ...]
    exact_match: bool
    full_metric: float
    corrupt_metric: float
    circuit_metric: float
    random_metric: float
    preserved_fraction: float
    minimality_damage: float
    completeness_gain: float
    random_margin: float
    passes: bool


def _require_finite_tensor(tensor: t.Tensor, name: str) -> None:
    if not t.isfinite(tensor).all():
        raise ValueError(f"{name} must be finite.")


def _require_finite_scalar(value: float, name: str) -> None:
    if not t.isfinite(t.tensor(value)).item():
        raise ValueError(f"{name} must be finite.")


def toy_acdc_graph() -> dict[str, object]:
    return {
        "edge_names": [
            "color_input -> answer_logit",
            "shape_input -> answer_logit",
            "background_input -> answer_logit",
            "position_input -> answer_logit",
        ],
        "clean_edge_values": t.tensor([2.0, 1.5, 0.2, -0.1]),
        "corrupt_edge_values": t.tensor([0.0, 0.0, 0.2, -0.1]),
        "ground_truth_edges": (
            "color_input -> answer_logit",
            "shape_input -> answer_logit",
        ),
        "same_size_random_edges": (
            "background_input -> answer_logit",
            "position_input -> answer_logit",
        ),
        "threshold": 0.35,
    }


def _validate_edge_values(
    clean_edge_values: t.Tensor,
    corrupt_edge_values: t.Tensor,
    edge_names: list[str],
) -> tuple[t.Tensor, t.Tensor]:
    clean = clean_edge_values.flatten().float()
    corrupt = corrupt_edge_values.flatten().float()
    if clean.numel() == 0:
        raise ValueError("edge values must be nonempty.")
    if clean.shape != corrupt.shape:
        raise ValueError("clean and corrupt edge values must have the same shape.")
    if clean.numel() != len(edge_names):
        raise ValueError("edge values and edge_names must align.")
    if any(not name for name in edge_names):
        raise ValueError("edge_names must be nonempty strings.")
    _require_finite_tensor(clean, "clean_edge_values")
    _require_finite_tensor(corrupt, "corrupt_edge_values")
    return clean, corrupt


def toy_edge_table(fixture: dict[str, object]) -> list[dict[str, object]]:
    clean = fixture["clean_edge_values"]
    corrupt = fixture["corrupt_edge_values"]
    rows = []
    for name, clean_value, corrupt_value in zip(fixture["edge_names"], clean, corrupt, strict=True):
        rows.append(
            {
                "edge": name,
                "clean contribution": float(clean_value.item()),
                "corrupt contribution": float(corrupt_value.item()),
                "delta if patched": float((clean_value - corrupt_value).item()),
                "ground truth": name in fixture["ground_truth_edges"],
            }
        )
    return rows


fixture = toy_acdc_graph()
display(toy_edge_table(fixture))


### Exercise - implement patch-recovery scores

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend 10-15 minutes on this exercise.
> ```

The circuit search starts with a metric. In the real model this is a target-minus-distractor logit difference; in the toy graph it is the sum of edge contributions. A patch score is the fraction of the clean-corrupt gap recovered by one patch.

<details>
<summary>Expected output</summary>

The controlled patch metrics should produce scores `[0.0, 0.5, 1.0]`, with best index `2`.

```text
All tests in `test_position_patching_helpers_score_recovery` passed!
All tests in `test_position_patching_helpers_reject_degenerate_inputs` passed!
```

</details>

<details>
<summary>Help - why normalize by the clean-corrupt gap?</summary>

Raw metric changes are hard to compare across tasks. Normalized recovery asks a direct causal question: how much of the clean behavior comes back when this component is patched?

</details>

<details>
<summary>Solution</summary>

```python
def answer_logit_diff(logits, *, positive_token_id, negative_token_id):
    diff = logits[..., positive_token_id] - logits[..., negative_token_id]
    return diff.float().mean().item()


def activation_patching_sweep(*, clean_metric, corrupt_metric, patched_metrics):
    denominator = clean_metric - corrupt_metric
    patch_scores = (patched_metrics.float() - corrupt_metric) / denominator
    best_index = int(patch_scores.argmax().item())
    return ActivationPatchingSweep(patch_scores, best_index, float(patch_scores[best_index].item()))
```

</details>


In [ ]:
def answer_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    if logits.ndim < 1:
        raise ValueError("logits must have a vocabulary dimension.")
    vocab_size = logits.shape[-1]
    if vocab_size == 0:
        raise ValueError("logits vocabulary dimension must be nonempty.")
    if positive_token_id == negative_token_id:
        raise ValueError("positive_token_id and negative_token_id must differ.")
    if not 0 <= positive_token_id < vocab_size:
        raise ValueError("positive_token_id is out of range.")
    if not 0 <= negative_token_id < vocab_size:
        raise ValueError("negative_token_id is out of range.")
    _require_finite_tensor(logits, "logits")
    diff = logits[..., positive_token_id] - logits[..., negative_token_id]
    return diff.float().mean().item()


def activation_patching_sweep(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metrics: t.Tensor,
) -> ActivationPatchingSweep:
    denominator = clean_metric - corrupt_metric
    for name, value in {
        "clean_metric": clean_metric,
        "corrupt_metric": corrupt_metric,
    }.items():
        _require_finite_scalar(value, name)
    if denominator == 0:
        raise ValueError("clean_metric and corrupt_metric must differ.")
    if patched_metrics.ndim != 1:
        raise ValueError("patched_metrics must be rank-1.")
    if patched_metrics.numel() == 0:
        raise ValueError("patched_metrics must be nonempty.")
    _require_finite_tensor(patched_metrics, "patched_metrics")
    patch_scores = (patched_metrics.float() - corrupt_metric) / denominator
    best_index = int(patch_scores.argmax().item())
    return ActivationPatchingSweep(
        patch_scores=patch_scores,
        best_index=best_index,
        best_score=float(patch_scores[best_index].item()),
    )


tests.test_position_patching_helpers_score_recovery(
    answer_logit_diff,
    activation_patching_sweep,
)
tests.test_position_patching_helpers_reject_degenerate_inputs(
    answer_logit_diff,
    activation_patching_sweep,
)


### Exercise - exact edge patching on a toy graph

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend 15-20 minutes on this exercise.
> ```

This is the toy oracle that the old 8.3 section lacked. The graph has four named edges, but only two matter. Your patching function should start from corrupt edge contributions and replace selected edges with their clean values.

<details>
<summary>Expected output</summary>

Exact single-edge patching should assign high scores to the color and shape edges, and the full toy evaluation should recover exactly those two edges.

```text
All tests in `test_toy_graph_exact_patching_recovers_known_circuit` passed!
All tests in `test_toy_graph_controls_fail_for_bad_circuits` passed!
```

</details>

<details>
<summary>Help - why start with a toy graph?</summary>

On the toy graph we know the ground-truth circuit. This lets us test ACDC-style pruning against an oracle before making any real-model claim.

</details>

<details>
<summary>Solution</summary>

```python
def patch_toy_graph_edges(clean_edge_values, corrupt_edge_values, edge_names, patched_edges):
    clean, corrupt = _validate_edge_values(clean_edge_values, corrupt_edge_values, edge_names)
    patched = corrupt.clone()
    name_to_index = {name: index for index, name in enumerate(edge_names)}
    for edge in patched_edges:
        patched[name_to_index[edge]] = clean[name_to_index[edge]]
    return float(patched.sum().item())
```

</details>


In [ ]:
def patch_toy_graph_edges(
    clean_edge_values: t.Tensor,
    corrupt_edge_values: t.Tensor,
    edge_names: list[str],
    patched_edges: tuple[str, ...] | list[str],
) -> float:
    clean, corrupt = _validate_edge_values(clean_edge_values, corrupt_edge_values, edge_names)
    name_to_index = {name: index for index, name in enumerate(edge_names)}
    patched = corrupt.clone()
    for edge in patched_edges:
        if edge not in name_to_index:
            raise ValueError(f"unknown edge: {edge}")
        index = name_to_index[edge]
        patched[index] = clean[index]
    return float(patched.sum().item())


def exact_toy_edge_patch_scores(
    clean_edge_values: t.Tensor,
    corrupt_edge_values: t.Tensor,
    edge_names: list[str],
) -> ActivationPatchingSweep:
    clean, corrupt = _validate_edge_values(clean_edge_values, corrupt_edge_values, edge_names)
    clean_metric = float(clean.sum().item())
    corrupt_metric = float(corrupt.sum().item())
    patched_metrics = t.tensor(
        [
            patch_toy_graph_edges(clean, corrupt, edge_names, [edge_name])
            for edge_name in edge_names
        ],
        dtype=t.float32,
    )
    return activation_patching_sweep(
        clean_metric=clean_metric,
        corrupt_metric=corrupt_metric,
        patched_metrics=patched_metrics,
    )


### Exercise - prune edges and evaluate the discovered toy circuit

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend 20 minutes on this exercise.
> ```

ACDC-style pruning keeps edges whose patch scores survive a threshold. A real circuit claim also needs a metric battery: faithfulness, minimality, completeness, and same-size random controls.

<details>
<summary>Expected output</summary>

The threshold `0.35` should keep the two ground-truth edges and reject both decoys. A threshold of `0.6` should visibly fail because it keeps only one true edge.

```text
All tests in `test_acdc_pruning_report_keeps_threshold_edges` passed!
All tests in `test_acdc_pruning_report_rejects_bad_scores_or_names` passed!
All tests in `test_toy_graph_exact_patching_recovers_known_circuit` passed!
All tests in `test_toy_graph_controls_fail_for_bad_circuits` passed!
```

</details>

<details>
<summary>Help - what each metric guards against</summary>

Faithfulness checks whether the kept circuit preserves behavior. Minimality checks whether removing kept edges hurts. Completeness checks whether adding omitted edges helps little. Random controls check whether any same-size circuit would have worked.

</details>

<details>
<summary>Solution</summary>

```python
def acdc_pruning_report(edge_scores, edge_names, *, threshold):
    kept = tuple(name for score, name in zip(edge_scores.flatten().tolist(), edge_names) if score >= threshold)
    removed = tuple(name for name in edge_names if name not in kept)
    return ACDCPruningReport(kept, removed, threshold, len(kept))
```

</details>


In [ ]:
def acdc_pruning_report(
    edge_scores: t.Tensor,
    edge_names: list[str],
    *,
    threshold: float,
) -> ACDCPruningReport:
    scores = edge_scores.flatten().float()
    if scores.numel() == 0:
        raise ValueError("edge_scores must be nonempty.")
    if scores.numel() != len(edge_names):
        raise ValueError("edge_scores and edge_names must align.")
    if any(not name for name in edge_names):
        raise ValueError("edge_names must be nonempty strings.")
    _require_finite_tensor(scores, "edge_scores")
    _require_finite_scalar(threshold, "threshold")
    kept = []
    removed = []
    for score, name in zip(scores.tolist(), edge_names, strict=True):
        if score >= threshold:
            kept.append(name)
        else:
            removed.append(name)
    return ACDCPruningReport(tuple(kept), tuple(removed), threshold, len(kept))


def circuit_faithfulness_report(
    *,
    full_metric: float,
    corrupt_metric: float,
    circuit_metric: float,
    min_preserved_fraction: float = 0.75,
) -> CircuitFaithfulnessReport:
    for name, value in {
        "full_metric": full_metric,
        "corrupt_metric": corrupt_metric,
        "circuit_metric": circuit_metric,
        "min_preserved_fraction": min_preserved_fraction,
    }.items():
        _require_finite_scalar(value, name)
    denominator = full_metric - corrupt_metric
    if denominator == 0:
        raise ValueError("full_metric and corrupt_metric must differ.")
    if min_preserved_fraction < 0:
        raise ValueError("min_preserved_fraction must be nonnegative.")
    preserved_fraction = (circuit_metric - corrupt_metric) / denominator
    return CircuitFaithfulnessReport(
        full_metric,
        corrupt_metric,
        circuit_metric,
        preserved_fraction,
        preserved_fraction >= min_preserved_fraction,
    )


def circuit_minimality_report(
    *,
    circuit_metric: float,
    ablated_metric: float,
    min_metric_damage: float = 0.5,
) -> CircuitMinimalityReport:
    for name, value in {
        "circuit_metric": circuit_metric,
        "ablated_metric": ablated_metric,
        "min_metric_damage": min_metric_damage,
    }.items():
        _require_finite_scalar(value, name)
    if min_metric_damage < 0:
        raise ValueError("min_metric_damage must be nonnegative.")
    metric_damage = circuit_metric - ablated_metric
    return CircuitMinimalityReport(
        circuit_metric,
        ablated_metric,
        metric_damage,
        metric_damage >= min_metric_damage,
    )


def circuit_completeness_report(
    *,
    circuit_metric: float,
    expanded_metric: float,
    max_omitted_node_gain: float = 0.2,
) -> CircuitCompletenessReport:
    for name, value in {
        "circuit_metric": circuit_metric,
        "expanded_metric": expanded_metric,
        "max_omitted_node_gain": max_omitted_node_gain,
    }.items():
        _require_finite_scalar(value, name)
    if max_omitted_node_gain < 0:
        raise ValueError("max_omitted_node_gain must be nonnegative.")
    omitted_node_gain = expanded_metric - circuit_metric
    return CircuitCompletenessReport(
        circuit_metric,
        expanded_metric,
        omitted_node_gain,
        omitted_node_gain <= max_omitted_node_gain,
    )


def random_circuit_baseline_report(
    *,
    circuit_metric: float,
    random_metric: float,
    min_margin: float = 0.5,
) -> RandomCircuitBaselineReport:
    for name, value in {
        "circuit_metric": circuit_metric,
        "random_metric": random_metric,
        "min_margin": min_margin,
    }.items():
        _require_finite_scalar(value, name)
    if min_margin < 0:
        raise ValueError("min_margin must be nonnegative.")
    margin = circuit_metric - random_metric
    return RandomCircuitBaselineReport(
        circuit_metric,
        random_metric,
        margin,
        margin >= min_margin,
    )


In [ ]:
def evaluate_toy_circuit(
    clean_edge_values: t.Tensor,
    corrupt_edge_values: t.Tensor,
    edge_names: list[str],
    ground_truth_edges: tuple[str, ...],
    random_edges: tuple[str, ...],
    *,
    threshold: float = 0.35,
) -> ToyCircuitEvaluationReport:
    clean, corrupt = _validate_edge_values(clean_edge_values, corrupt_edge_values, edge_names)
    sweep = exact_toy_edge_patch_scores(clean, corrupt, edge_names)
    pruning = acdc_pruning_report(sweep.patch_scores, edge_names, threshold=threshold)
    discovered_edges = pruning.kept_edges
    full_metric = float(clean.sum().item())
    corrupt_metric = float(corrupt.sum().item())
    circuit_metric = patch_toy_graph_edges(clean, corrupt, edge_names, discovered_edges)
    random_metric = patch_toy_graph_edges(clean, corrupt, edge_names, random_edges)

    if not discovered_edges:
        ablated_metric = corrupt_metric
    else:
        ablated_metric = max(
            patch_toy_graph_edges(
                clean,
                corrupt,
                edge_names,
                tuple(edge for edge in discovered_edges if edge != edge_to_remove),
            )
            for edge_to_remove in discovered_edges
        )

    omitted_edges = [edge for edge in edge_names if edge not in discovered_edges]
    if omitted_edges:
        top_omitted = max(
            omitted_edges,
            key=lambda edge: float(sweep.patch_scores[edge_names.index(edge)].item()),
        )
        expanded_metric = patch_toy_graph_edges(
            clean,
            corrupt,
            edge_names,
            (*discovered_edges, top_omitted),
        )
    else:
        expanded_metric = circuit_metric

    faithfulness = circuit_faithfulness_report(
        full_metric=full_metric,
        corrupt_metric=corrupt_metric,
        circuit_metric=circuit_metric,
        min_preserved_fraction=0.99,
    )
    minimality = circuit_minimality_report(
        circuit_metric=circuit_metric,
        ablated_metric=ablated_metric,
        min_metric_damage=1.0,
    )
    completeness = circuit_completeness_report(
        circuit_metric=circuit_metric,
        expanded_metric=expanded_metric,
        max_omitted_node_gain=1e-6,
    )
    random_baseline = random_circuit_baseline_report(
        circuit_metric=circuit_metric,
        random_metric=random_metric,
        min_margin=1.0,
    )
    exact_match = set(discovered_edges) == set(ground_truth_edges)
    passes = (
        exact_match
        and faithfulness.passes_faithfulness
        and minimality.passes_minimality
        and completeness.passes_completeness
        and random_baseline.circuit_beats_random
    )
    return ToyCircuitEvaluationReport(
        ground_truth_edges=tuple(ground_truth_edges),
        discovered_edges=tuple(discovered_edges),
        exact_match=exact_match,
        full_metric=full_metric,
        corrupt_metric=corrupt_metric,
        circuit_metric=circuit_metric,
        random_metric=random_metric,
        preserved_fraction=faithfulness.preserved_fraction,
        minimality_damage=minimality.metric_damage,
        completeness_gain=completeness.omitted_node_gain,
        random_margin=random_baseline.margin,
        passes=passes,
    )


tests.test_acdc_pruning_report_keeps_threshold_edges(acdc_pruning_report)
tests.test_acdc_pruning_report_rejects_bad_scores_or_names(acdc_pruning_report)
tests.test_toy_graph_exact_patching_recovers_known_circuit(
    toy_acdc_graph,
    exact_toy_edge_patch_scores,
    evaluate_toy_circuit,
)
tests.test_toy_graph_controls_fail_for_bad_circuits(
    patch_toy_graph_edges,
    evaluate_toy_circuit,
)


### Exercise - held-out templates and exact-vs-approximate comparison

> ```yaml
> Difficulty: medium
> Importance: medium
>
> You should spend 15-20 minutes on this exercise.
> ```

The toy graph tells us whether exact patching works. Real circuit discovery often compares exact scores with cheaper approximations, and checks that the selected circuit generalizes across prompt templates.

<details>
<summary>Expected output</summary>

OOD should fail when any template drops below threshold. Approximate scores should pass when they recover exact top-k edges and fail when they rank decoys above true edges.

```text
All tests in `test_ood_template_report_tracks_worst_template` passed!
All tests in `test_circuit_method_comparison_report_matches_exact_patching` passed!
```

</details>

<details>
<summary>Help - averages can hide broken templates</summary>

For circuit validation, the worst template often matters more than the mean. A circuit that only works on one prompt wording is a fragile artifact.

</details>

<details>
<summary>Solution</summary>

```python
def ood_template_report(logits, answer_ids, template_ids, *, min_accuracy=0.75):
    predictions = logits.argmax(dim=-1)
    per_template = {}
    for template_id in template_ids.unique(sorted=True):
        mask = template_ids.eq(template_id)
        per_template[int(template_id.item())] = predictions[mask].eq(answer_ids[mask]).float().mean().item()
    worst = min(per_template.values())
    return OODTemplateReport(per_template, worst, worst >= min_accuracy)
```

</details>


In [ ]:
def ood_template_report(
    logits: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
    *,
    min_accuracy: float = 0.75,
) -> OODTemplateReport:
    if logits.shape[:-1] != answer_ids.shape:
        raise ValueError("answer_ids must match logits leading dimensions.")
    if answer_ids.shape != template_ids.shape:
        raise ValueError("answer_ids and template_ids must match.")
    if answer_ids.numel() == 0:
        raise ValueError("answer_ids must be nonempty.")
    if logits.shape[-1] == 0:
        raise ValueError("logits vocabulary dimension must be nonempty.")
    if not 0.0 <= min_accuracy <= 1.0:
        raise ValueError("min_accuracy must be between 0 and 1.")
    _require_finite_tensor(logits, "logits")
    if ((answer_ids < 0) | (answer_ids >= logits.shape[-1])).any():
        raise ValueError("answer_ids must be valid vocabulary indices.")

    predictions = logits.argmax(dim=-1)
    per_template: dict[int, float] = {}
    for template_id in template_ids.unique(sorted=True):
        mask = template_ids.eq(template_id)
        accuracy = predictions[mask].eq(answer_ids[mask]).float().mean().item()
        per_template[int(template_id.item())] = accuracy
    worst_accuracy = min(per_template.values()) if per_template else 0.0
    return OODTemplateReport(per_template, worst_accuracy, worst_accuracy >= min_accuracy)


def _top_edge_names(scores: t.Tensor, edge_names: list[str], *, top_k: int) -> tuple[str, ...]:
    flat_scores = scores.flatten().float()
    if flat_scores.numel() == 0:
        raise ValueError("scores must be nonempty.")
    if flat_scores.numel() != len(edge_names):
        raise ValueError("scores and edge_names must align.")
    _require_finite_tensor(flat_scores, "scores")
    k = min(top_k, flat_scores.numel())
    top_indices = flat_scores.topk(k=k).indices.tolist()
    return tuple(edge_names[int(index)] for index in top_indices)


def _pearson_correlation(left: t.Tensor, right: t.Tensor) -> float:
    left = left.flatten().float()
    right = right.flatten().float()
    if left.shape != right.shape:
        raise ValueError("score tensors must have matching shapes.")
    if left.numel() < 2:
        raise ValueError("at least two scores are required for correlation.")
    _require_finite_tensor(left, "left")
    _require_finite_tensor(right, "right")
    left_centered = left - left.mean()
    right_centered = right - right.mean()
    denominator = left_centered.norm() * right_centered.norm()
    if float(denominator.item()) == 0.0:
        return 0.0
    return float((left_centered @ right_centered / denominator).item())


def circuit_method_comparison_report(
    exact_scores: t.Tensor,
    method_scores: dict[str, t.Tensor],
    edge_names: list[str],
    *,
    top_k: int,
    min_topk_overlap: float = 0.5,
    min_score_correlation: float = 0.5,
) -> CircuitMethodComparisonReport:
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    if not 0.0 <= min_topk_overlap <= 1.0:
        raise ValueError("min_topk_overlap must be between 0 and 1.")
    if not -1.0 <= min_score_correlation <= 1.0:
        raise ValueError("min_score_correlation must be between -1 and 1.")
    if not method_scores:
        raise ValueError("method_scores must contain at least one method.")
    exact_flat = exact_scores.flatten().float()
    if exact_flat.numel() == 0:
        raise ValueError("exact_scores must be nonempty.")
    if exact_flat.numel() != len(edge_names):
        raise ValueError("exact_scores and edge_names must align.")
    _require_finite_tensor(exact_flat, "exact_scores")

    exact_top_edges = _top_edge_names(exact_flat, edge_names, top_k=top_k)
    exact_top_set = set(exact_top_edges)
    method_top_edges: dict[str, tuple[str, ...]] = {}
    topk_overlap: dict[str, float] = {}
    score_correlations: dict[str, float] = {}
    circuit_sizes: dict[str, int] = {"exact": len(exact_top_edges)}

    for method_name, scores in method_scores.items():
        method_flat = scores.flatten().float()
        if method_flat.shape != exact_flat.shape:
            raise ValueError(f"{method_name} scores must match exact_scores shape.")
        _require_finite_tensor(method_flat, f"{method_name} scores")
        top_edges = _top_edge_names(method_flat, edge_names, top_k=top_k)
        overlap = len(exact_top_set.intersection(top_edges)) / len(exact_top_edges)
        method_top_edges[method_name] = top_edges
        topk_overlap[method_name] = overlap
        score_correlations[method_name] = _pearson_correlation(exact_flat, method_flat)
        circuit_sizes[method_name] = len(top_edges)

    best_matching_method = max(
        method_scores,
        key=lambda name: (topk_overlap[name], score_correlations[name]),
    )
    passes_comparison = all(
        topk_overlap[name] >= min_topk_overlap
        and score_correlations[name] >= min_score_correlation
        for name in method_scores
    )
    return CircuitMethodComparisonReport(
        exact_top_edges,
        method_top_edges,
        topk_overlap,
        score_correlations,
        circuit_sizes,
        best_matching_method,
        passes_comparison,
    )


tests.test_ood_template_report_tracks_worst_template(ood_template_report)
tests.test_ood_template_report_rejects_degenerate_inputs(ood_template_report)
tests.test_circuit_method_comparison_report_matches_exact_patching(
    circuit_method_comparison_report,
)
tests.test_circuit_method_comparison_report_rejects_bad_inputs(
    circuit_method_comparison_report,
)


## Signature Result - recover the toy circuit exactly

The toy graph is the ground-truth check. You should not trust a real-model circuit section until this plot and table make sense.

<details>
<summary>Expected output</summary>

The edge-score table should show two nonzero scores, and the metric plot should show the discovered circuit at the clean metric while the same-size random circuit stays at the corrupt metric.

</details>

<details>
<summary>Interpreting the result</summary>

The discovered circuit is meaningful because it passes all four checks at once: exact recovery, faithfulness, minimality, and random baseline. Any one of these alone is weaker.

</details>


In [ ]:
def toy_signature_result() -> dict[str, Any]:
    fixture = toy_acdc_graph()
    sweep = exact_toy_edge_patch_scores(
        fixture["clean_edge_values"],
        fixture["corrupt_edge_values"],
        fixture["edge_names"],
    )
    report = evaluate_toy_circuit(
        fixture["clean_edge_values"],
        fixture["corrupt_edge_values"],
        fixture["edge_names"],
        fixture["ground_truth_edges"],
        fixture["same_size_random_edges"],
        threshold=fixture["threshold"],
    )
    rows = []
    for name, score in zip(fixture["edge_names"], sweep.patch_scores, strict=True):
        rows.append(
            {
                "edge": name,
                "patch score": round(float(score.item()), 3),
                "kept": name in report.discovered_edges,
                "ground truth": name in report.ground_truth_edges,
            }
        )
    thresholds = [0.0, 0.2, 0.35, 0.5, 0.6]
    threshold_rows = []
    for threshold in thresholds:
        trial = evaluate_toy_circuit(
            fixture["clean_edge_values"],
            fixture["corrupt_edge_values"],
            fixture["edge_names"],
            fixture["ground_truth_edges"],
            fixture["same_size_random_edges"],
            threshold=threshold,
        )
        threshold_rows.append(
            {
                "threshold": threshold,
                "discovered": trial.discovered_edges,
                "exact match": trial.exact_match,
                "passes all controls": trial.passes,
            }
        )
    return {"rows": rows, "threshold_rows": threshold_rows, "report": report}


toy_signature = toy_signature_result()
display(toy_signature["rows"])
display(toy_signature["threshold_rows"])
display(toy_signature["report"].__dict__)

assert toy_signature["report"].exact_match
assert toy_signature["report"].passes


In [ ]:
report = toy_signature["report"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].bar(
    [row["edge"].split(" -> ")[0] for row in toy_signature["rows"]],
    [row["patch score"] for row in toy_signature["rows"]],
    color=["#16a34a" if row["kept"] else "#94a3b8" for row in toy_signature["rows"]],
)
axes[0].axhline(fixture["threshold"], color="#334155", linestyle="--", linewidth=1)
axes[0].set_ylim(0, 0.7)
axes[0].set_title("Toy edge patch scores")
axes[0].set_ylabel("normalized recovery")
axes[0].tick_params(axis="x", rotation=15)

metric_names = ["corrupt", "random", "circuit", "clean"]
metric_values = [
    report.corrupt_metric,
    report.random_metric,
    report.circuit_metric,
    report.full_metric,
]
axes[1].bar(metric_names, metric_values, color=["#64748b", "#f97316", "#16a34a", "#2563eb"])
axes[1].set_title("Metric recovery and controls")
axes[1].set_ylabel("toy answer metric")
for i, value in enumerate(metric_values):
    axes[1].text(i, value + 0.08, f"{value:.1f}", ha="center")

fig.tight_layout()
plt.show()


### Exercise - whole-notebook smoke contract

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend 5 minutes on this exercise after all subfunction tests pass.
> ```

The whole-notebook contract checks that the visible toy circuit, metric battery, OOD report, and method comparison compose correctly.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Help - use this only after subfunction tests pass</summary>

If this fails, inspect the earlier tests first. Whole-notebook tests are for integration mistakes, not for debugging a matrix multiply or edge-name mismatch.

</details>


In [ ]:
def acdc_pruning_smoke_test() -> dict:
    scores = t.tensor([0.9, 0.2, 0.7])
    names = ["name-mover", "backup", "negative"]
    return acdc_pruning_report(scores, names, threshold=0.5).__dict__


def toy_circuit_smoke_test() -> dict:
    fixture = toy_acdc_graph()
    sweep = exact_toy_edge_patch_scores(
        fixture["clean_edge_values"],
        fixture["corrupt_edge_values"],
        fixture["edge_names"],
    )
    report = evaluate_toy_circuit(
        fixture["clean_edge_values"],
        fixture["corrupt_edge_values"],
        fixture["edge_names"],
        fixture["ground_truth_edges"],
        fixture["same_size_random_edges"],
        threshold=fixture["threshold"],
    )
    return {
        "edge_names": fixture["edge_names"],
        "patch_scores": [float(score) for score in sweep.patch_scores.tolist()],
        **report.__dict__,
    }


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    exact = t.tensor([0.95, 0.8, 0.15, 0.05])
    eap_ig = t.tensor([0.9, 0.7, 0.2, 0.1])
    names = ["name-mover", "backup-name-mover", "mlp-noise", "wrong-position"]
    return {
        "toy_circuit": toy_circuit_smoke_test(),
        "acdc": acdc_pruning_smoke_test(),
        "faithfulness": circuit_faithfulness_report(
            full_metric=3.0,
            corrupt_metric=-1.0,
            circuit_metric=2.2,
            min_preserved_fraction=0.75,
        ).__dict__,
        "minimality": circuit_minimality_report(
            circuit_metric=2.2,
            ablated_metric=0.5,
            min_metric_damage=1.0,
        ).__dict__,
        "completeness": circuit_completeness_report(
            circuit_metric=2.2,
            expanded_metric=2.35,
            max_omitted_node_gain=0.2,
        ).__dict__,
        "random_baseline": random_circuit_baseline_report(
            circuit_metric=2.2,
            random_metric=0.8,
            min_margin=1.0,
        ).__dict__,
        "ood": ood_template_report(
            t.tensor([[2.0, 0.0], [0.0, 2.0], [2.0, 0.0], [0.0, 2.0]]),
            t.tensor([0, 1, 0, 1]),
            t.tensor([0, 0, 1, 1]),
            min_accuracy=1.0,
        ).__dict__,
        "method_comparison": circuit_method_comparison_report(
            exact,
            {"eap_ig": eap_ig},
            names,
            top_k=2,
            min_topk_overlap=1.0,
            min_score_correlation=0.9,
        ).__dict__,
    }


tests.test_notebook_contract(run_smoke_test)


## Real-Model Fragment - GELU-1L final residual position

Now move from the toy oracle to a narrow real-model fragment. This is not a full ACDC replication. The goal is to check whether exact residual-position patching on a small public TransformerLens model passes the same metric discipline.

<details>
<summary>Expected output</summary>

The plot should show one dominant final-position patch score on the primary prompt pair, held-out template recoveries near `1.0`, random/wrong-position recoveries near `0.0`, and peak VRAM below `1 GB`.

</details>

<details>
<summary>Interpreting the result</summary>

This result is deliberately scoped: it validates the metric battery on a small final-position fragment. It does not claim full ACDC, IOI, greater-than circuits, or multi-node path discovery.

</details>


In [ ]:
TL_GELU1L_MODEL_NAME = "gelu-1l"
TL_GELU1L_HF_ID = "NeelNanda/GELU_1L512W_C4_Code"
TL_GELU1L_REVISION = "bddc0e332f0ae84279e6a6a45d91b314899e1603"
TL_GELU1L_TOKENIZER_ID = "NeelNanda/gpt-neox-tokenizer-digits"
TL_GELU1L_TOKENIZER_REVISION = "0f6671571a20be9756b9991d978047c03b75e749"
TL_PATCH_HOOK_NAME = "blocks.0.hook_resid_post"
TL_BNB_CUDA_OVERRIDE = "130"
TL_PRIMARY_CLEAN_PROMPT = "The cat sat on the"
TL_PRIMARY_CORRUPT_PROMPT = "The bird flew over the"
TL_TEMPLATE_PAIRS = [
    ("To make tea, boil the", "To make bread, bake the"),
    ("The recipe calls for sugar and", "The recipe calls for salt and"),
    ("The chef cooked a", "The teacher taught a"),
]


def _load_gelu1l_model_on_cuda():
    if not t.cuda.is_available():
        raise RuntimeError("CUDA is required for the real GELU-1L fragment.")
    os.environ.setdefault("BNB_CUDA_VERSION", TL_BNB_CUDA_OVERRIDE)
    logging.getLogger("bitsandbytes.cextension").setLevel(logging.ERROR)
    from transformer_lens import HookedTransformer
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        TL_GELU1L_TOKENIZER_ID,
        revision=TL_GELU1L_TOKENIZER_REVISION,
    )
    return HookedTransformer.from_pretrained(
        TL_GELU1L_MODEL_NAME,
        device="cuda",
        dtype="float32",
        revision=TL_GELU1L_REVISION,
        tokenizer=tokenizer,
    )


def _patched_metric(
    model,
    corrupt_tokens: t.Tensor,
    clean_cache: dict[str, t.Tensor],
    positions: list[int],
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    def patch_hook(activation: t.Tensor, hook) -> t.Tensor:
        patched = activation.clone()
        for position in positions:
            patched[:, position, :] = clean_cache[TL_PATCH_HOOK_NAME][:, position, :]
        return patched

    with t.inference_mode():
        patched_logits = model.run_with_hooks(
            corrupt_tokens,
            fwd_hooks=[(TL_PATCH_HOOK_NAME, patch_hook)],
        )
    return answer_logit_diff(
        patched_logits[0, -1],
        positive_token_id=positive_token_id,
        negative_token_id=negative_token_id,
    )


def _real_patch_sweep(model, clean_prompt: str, corrupt_prompt: str) -> dict[str, Any]:
    clean_tokens = model.to_tokens(clean_prompt)
    corrupt_tokens = model.to_tokens(corrupt_prompt)
    if clean_tokens.shape != corrupt_tokens.shape:
        raise RuntimeError(f"token shapes differ: {clean_tokens.shape} vs {corrupt_tokens.shape}")

    with t.inference_mode():
        clean_logits, clean_cache = model.run_with_cache(
            clean_tokens,
            names_filter=lambda name: name == TL_PATCH_HOOK_NAME,
        )
        corrupt_logits = model(corrupt_tokens)

    clean_final_logits = clean_logits[0, -1]
    corrupt_final_logits = corrupt_logits[0, -1]
    clean_top_tokens = clean_final_logits.topk(5).indices.tolist()
    corrupt_top_tokens = corrupt_final_logits.topk(5).indices.tolist()
    target_token_id = clean_top_tokens[0]
    distractor_token_id = next(
        token_id
        for token_id in [*corrupt_top_tokens, *clean_top_tokens[1:]]
        if token_id != target_token_id
    )
    clean_metric = answer_logit_diff(
        clean_final_logits,
        positive_token_id=target_token_id,
        negative_token_id=distractor_token_id,
    )
    corrupt_metric = answer_logit_diff(
        corrupt_final_logits,
        positive_token_id=target_token_id,
        negative_token_id=distractor_token_id,
    )

    sequence_length = int(clean_tokens.shape[1])
    patched_metrics = [
        _patched_metric(
            model,
            corrupt_tokens,
            clean_cache,
            [position],
            positive_token_id=target_token_id,
            negative_token_id=distractor_token_id,
        )
        for position in range(sequence_length)
    ]
    sweep = activation_patching_sweep(
        clean_metric=clean_metric,
        corrupt_metric=corrupt_metric,
        patched_metrics=t.tensor(patched_metrics, device=clean_logits.device),
    )
    target_position = sequence_length - 1
    non_target_positions = [position for position in range(sequence_length) if position != target_position]
    random_position = non_target_positions[0]
    omitted_scores = sweep.patch_scores.clone()
    omitted_scores[target_position] = -t.inf
    top_omitted_position = int(omitted_scores.argmax().item())
    expanded_positions = [target_position, top_omitted_position]
    return {
        "clean_prompt": clean_prompt,
        "corrupt_prompt": corrupt_prompt,
        "sequence_length": sequence_length,
        "target_position": target_position,
        "random_position": random_position,
        "top_omitted_position": top_omitted_position,
        "target_token": model.to_string(target_token_id),
        "distractor_token": model.to_string(distractor_token_id),
        "clean_metric": clean_metric,
        "corrupt_metric": corrupt_metric,
        "clean_corrupt_gap": clean_metric - corrupt_metric,
        "patch_scores": sweep.patch_scores,
        "best_position": sweep.best_index,
        "best_score": sweep.best_score,
        "circuit_metric": patched_metrics[target_position],
        "random_metric": patched_metrics[random_position],
        "top_omitted_metric": patched_metrics[top_omitted_position],
        "expanded_metric": _patched_metric(
            model,
            corrupt_tokens,
            clean_cache,
            expanded_positions,
            positive_token_id=target_token_id,
            negative_token_id=distractor_token_id,
        ),
    }


def run_transformerlens_acdc_preflight(max_vram_gb: float = 24.0) -> dict[str, Any]:
    t.cuda.reset_peak_memory_stats()
    model = _load_gelu1l_model_on_cuda()
    model.eval()

    primary = _real_patch_sweep(model, TL_PRIMARY_CLEAN_PROMPT, TL_PRIMARY_CORRUPT_PROMPT)
    edge_names = [f"position_{index}" for index in range(primary["sequence_length"])]
    pruning = acdc_pruning_report(primary["patch_scores"], edge_names, threshold=0.5)
    faithfulness = circuit_faithfulness_report(
        full_metric=primary["clean_metric"],
        corrupt_metric=primary["corrupt_metric"],
        circuit_metric=primary["circuit_metric"],
        min_preserved_fraction=0.99,
    )
    minimality = circuit_minimality_report(
        circuit_metric=primary["circuit_metric"],
        ablated_metric=primary["corrupt_metric"],
        min_metric_damage=1.0,
    )
    completeness = circuit_completeness_report(
        circuit_metric=primary["circuit_metric"],
        expanded_metric=primary["expanded_metric"],
        max_omitted_node_gain=1e-5,
    )
    random_baseline = random_circuit_baseline_report(
        circuit_metric=primary["circuit_metric"],
        random_metric=primary["random_metric"],
        min_margin=1.0,
    )

    template_reports = [_real_patch_sweep(model, clean, corrupt) for clean, corrupt in TL_TEMPLATE_PAIRS]
    template_recoveries = [
        (template["circuit_metric"] - template["corrupt_metric"]) / template["clean_corrupt_gap"]
        for template in template_reports
    ]
    template_random_recoveries = [
        (template["random_metric"] - template["corrupt_metric"]) / template["clean_corrupt_gap"]
        for template in template_reports
    ]
    passes_ood = all(
        recovery >= 0.99
        and random_recovery <= 1e-4
        and template["best_position"] == template["target_position"]
        for recovery, random_recovery, template in zip(
            template_recoveries,
            template_random_recoveries,
            template_reports,
            strict=True,
        )
    )

    t.cuda.synchronize()
    peak_vram_gb = t.cuda.max_memory_allocated() / 1024**3
    return {
        "cuda_available": True,
        "torch_version": t.__version__,
        "cuda_version": t.version.cuda,
        "device": t.cuda.get_device_name(0),
        "preflight_passed": (
            pruning.kept_edges == (f"position_{primary['target_position']}",)
            and primary["best_position"] == primary["target_position"]
            and primary["best_score"] >= 0.99
            and faithfulness.passes_faithfulness
            and minimality.passes_minimality
            and completeness.passes_completeness
            and random_baseline.circuit_beats_random
            and passes_ood
            and peak_vram_gb <= max_vram_gb
        ),
        "model_name": TL_GELU1L_MODEL_NAME,
        "clean_prompt": TL_PRIMARY_CLEAN_PROMPT,
        "corrupt_prompt": TL_PRIMARY_CORRUPT_PROMPT,
        "target_token": primary["target_token"],
        "distractor_token": primary["distractor_token"],
        "target_position": primary["target_position"],
        "patch_scores_by_position": [float(score) for score in primary["patch_scores"].tolist()],
        "best_position": primary["best_position"],
        "best_score": primary["best_score"],
        "kept_edges": list(pruning.kept_edges),
        "preserved_fraction": faithfulness.preserved_fraction,
        "minimality_metric_damage": minimality.metric_damage,
        "omitted_node_gain": completeness.omitted_node_gain,
        "random_baseline_margin": random_baseline.margin,
        "template_recoveries": [float(value) for value in template_recoveries],
        "template_random_recoveries": [float(value) for value in template_random_recoveries],
        "passes_ood": passes_ood,
        "peak_vram_gb": peak_vram_gb,
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict[str, Any]:
    return run_transformerlens_acdc_preflight(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict[str, Any]:
    return run_gpu_test(max_vram_gb=max_vram_gb)


In [ ]:
real_result = run_gpu_test(max_vram_gb=24.0)

print(f"device: {real_result['device']}")
print(f"model: {real_result['model_name']}")
print(f"clean prompt: {real_result['clean_prompt']!r}")
print(f"corrupt prompt: {real_result['corrupt_prompt']!r}")
print(f"target / distractor: {real_result['target_token']!r} / {real_result['distractor_token']!r}")
print(f"peak VRAM GB: {real_result['peak_vram_gb']:.3f}")

display({key: real_result[key] for key in [
    "preflight_passed",
    "best_position",
    "target_position",
    "kept_edges",
    "preserved_fraction",
    "minimality_metric_damage",
    "omitted_node_gain",
    "random_baseline_margin",
    "passes_ood",
]})

assert real_result["preflight_passed"]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

positions = list(range(len(real_result["patch_scores_by_position"])))
axes[0].bar(
    [str(pos) for pos in positions],
    real_result["patch_scores_by_position"],
    color=["#16a34a" if pos == real_result["target_position"] else "#94a3b8" for pos in positions],
)
axes[0].axhline(0.5, color="#334155", linestyle="--", linewidth=1)
axes[0].set_ylim(0, 1.1)
axes[0].set_title("GELU-1L exact patch recovery")
axes[0].set_xlabel("residual position")
axes[0].set_ylabel("normalized recovery")

x = list(range(len(real_result["template_recoveries"])))
width = 0.35
axes[1].bar(
    [i - width / 2 for i in x],
    real_result["template_recoveries"],
    width=width,
    label="final-position circuit",
    color="#7c3aed",
)
axes[1].bar(
    [i + width / 2 for i in x],
    real_result["template_random_recoveries"],
    width=width,
    label="same-size wrong position",
    color="#f97316",
)
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Held-out template controls")
axes[1].set_xlabel("held-out pair")
axes[1].set_ylabel("normalized recovery")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()


## Try It Yourself - change the threshold or prompt pair

Change `PLAY_THRESHOLD` to see when the toy circuit becomes too small or too large. On a GPU machine, you can also change `PLAY_CLEAN_PROMPT` and `PLAY_CORRUPT_PROMPT` to another same-token-length prompt pair and rerun the real fragment helper.

<details>
<summary>Interpretation checklist</summary>

- Does the threshold recover exactly the known toy circuit?
- Does the same-size random circuit stay low?
- On the real prompt pair, is one position dominant or are scores diffuse?
- If the real scores are diffuse, which metric would fail first: faithfulness, minimality, completeness, or random baseline?

</details>


In [ ]:
PLAY_THRESHOLD = 0.35  # Try 0.2, 0.35, 0.6.
PLAY_CLEAN_PROMPT = "The cat sat on the"
PLAY_CORRUPT_PROMPT = "The bird flew over the"

fixture = toy_acdc_graph()
play_report = evaluate_toy_circuit(
    fixture["clean_edge_values"],
    fixture["corrupt_edge_values"],
    fixture["edge_names"],
    fixture["ground_truth_edges"],
    fixture["same_size_random_edges"],
    threshold=PLAY_THRESHOLD,
)
display(play_report.__dict__)

# For the real-model play path, edit the prompt constants above and rerun `run_gpu_test` after
# modifying `TL_PRIMARY_CLEAN_PROMPT` / `TL_PRIMARY_CORRUPT_PROMPT` in the helper cell.


## Limitations

This notebook now teaches real ACDC-style discipline, but the real-model result remains a scoped final-position fragment. It is not a full IOI or greater-than ACDC replication, does not discover multi-layer path circuits, and does not claim that final-position localization is a complete circuit. The strong claim is the toy result: exact edge patching recovers a known two-edge circuit and rejects decoys under explicit controls. The GELU-1L section is a mechanics preflight that applies the same metric battery to one small public checkpoint.
